In [ ]:
!pip install pdbparser biopython mygene openpyxl freesasa py3Dmol -q
import sys
!{sys.executable} -m pip install pymol-open-source -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.9/161.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 58.1 MB/s eta 0:00:00


In [ ]:
import pymol
from pymol import cmd
import matplotlib.colors as mcolors
import pandas as pd

# --- Parameters ---
pdb_file = 'Mouse STING homodimer(ranked_0).pdb'  # Ensure this file is in the same directory
output_png = 'residue_colored.png'

# Define your residue color mapping { "residue_id": "hex_color" }
excel_file = 'variant_clusters_cGAMP_4T1.csv'
residue_df = pd.read_csv(excel_file)

# Find the start and end indices
start_idx = residue_df[residue_df['variant'] == 'P316'].index[0]
end_idx = residue_df[residue_df['variant'] == 'Y239'].index[0]

# Slice the dataframe (this includes both start and end rows)
residue_df = residue_df.loc[start_idx:end_idx]
residue_colors = dict(zip(residue_df['residue'].astype(str), residue_df['color']))

# --- Initialize PyMOL ---
# '-cq' runs PyMOL in command-line quiet mode
pymol.finish_launching(['pymol', '-cq'])
cmd.reinitialize()

# Load structure and set base style
cmd.load(pdb_file, 'prot')
cmd.hide('everything', 'all')
cmd.show('cartoon', 'prot')
cmd.color('gray70', 'prot')

# Visual clean-up (No shadows, white background)
cmd.bg_color('white')
cmd.set('ray_shadows', 0)
cmd.set('depth_cue', 0)
cmd.set('ray_opaque_background', 0)

# --- Apply Sphere Coloring ---
for resi, hex_color in residue_colors.items():
    # Define a unique name for the color in PyMOL
    color_name = f'color_{resi}'
    rgb = mcolors.to_rgb(hex_color)
    cmd.set_color(color_name, rgb)

    # Select and show spheres
    selection = f'resi {resi}'
    cmd.show('spheres', selection)
    cmd.color(color_name, selection)

# Center the view on the highlighted spheres
cmd.zoom('spheres', buffer=5)

cmd.set_view([0.479985863,   -0.019433912,   -0.877059817,
     0.655858099,   -0.656033754,    0.373463154,
    -0.582640409,   -0.754482031,   -0.302139550,
     0.000000000,    0.000000000, -306.655059814,
     0.225017548,   -0.535675049,   -0.549560547,
   261.217132568,  352.092956543,  -20.000000000 ])

# --- Render and Save ---
cmd.ray(1200, 900)
cmd.png(output_png, dpi=300)
print(f'Successfully saved image to {output_png}')


Successfully saved image to residue_colored.png
